# 03 - Organização e normalização

Constrói os formatos long, matricial e tensorial para as duas planilhas, junta HARMÔNICAS e INTERHARMÔNICAS por timestamp mais próximo (mesmo rótulo) e demonstra as quatro normalizações disponíveis. Os artefatos são salvos em `data/processed/`.

In [1]:
from harmonic_signatures.io.cache import read_workbook_cached
from harmonic_signatures.config import (
    DEFAULT_WORKBOOK, INTERIM_DIR, PROCESSED_DIR, DEFAULT_JOIN_TOLERANCE_SECONDS,
)
from harmonic_signatures.organize.long_format import to_long
from harmonic_signatures.organize.matrix_format import to_matrix
from harmonic_signatures.organize.tensor_format import build_sequence_tensor
from harmonic_signatures.organize.join import nearest_timestamp_join

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
result = read_workbook_cached(DEFAULT_WORKBOOK, INTERIM_DIR)

## Formato long

In [2]:
harm_long = to_long(result.harmonicas)
inter_long = to_long(result.interharmonicas)
print('harm_long:', harm_long.shape)
print('inter_long:', inter_long.shape)

harm_long.to_parquet(PROCESSED_DIR / 'harmonicas_long.parquet', index=False)
inter_long.to_parquet(PROCESSED_DIR / 'interharmonicas_long.parquet', index=False)

harm_long: (12987666, 11)
inter_long: (12168624, 11)


## Formato matricial (X, y) e junção por timestamp

In [3]:
harm_matrix = to_matrix(harm_long)
inter_matrix = to_matrix(inter_long)
print('harm_matrix:', harm_matrix.shape)
print('inter_matrix:', inter_matrix.shape)

joined = nearest_timestamp_join(
    harm_matrix, inter_matrix, tolerance_seconds=DEFAULT_JOIN_TOLERANCE_SECONDS,
)
print('joined:', joined.shape, '| taxa de pareamento:', joined['matched'].mean())
joined.to_parquet(PROCESSED_DIR / 'combined_matrix.parquet', index=False)

harm_matrix: (19501, 669)
inter_matrix: (19501, 627)
joined: (19501, 1297) | taxa de pareamento: 1.0


## Formato tensorial (amostras × ordem × canais)

In [4]:
import numpy as np

harm_bundle = build_sequence_tensor(harm_long)
print('tensor:', harm_bundle.tensor.shape, '| static:', harm_bundle.static.shape)
print('canais:', harm_bundle.channel_names)

np.savez(
    PROCESSED_DIR / 'harmonicas_tensor.npz',
    tensor=harm_bundle.tensor,
    y=harm_bundle.y,
    sample_ids=np.array(harm_bundle.sample_ids),
    channel_names=np.array(harm_bundle.channel_names),
    order_values=np.array(harm_bundle.order_values),
)
harm_bundle.static.to_parquet(PROCESSED_DIR / 'harmonicas_static_features.parquet')

tensor: (19501, 51, 12) | static: (19501, 60)
canais: ['H_Ia_avg', 'H_Ia_max', 'H_Ib_avg', 'H_Ib_max', 'H_Ic_avg', 'H_Ic_max', 'H_Va_avg', 'H_Va_max', 'H_Vb_avg', 'H_Vb_max', 'H_Vc_avg', 'H_Vc_max']


## Normalizações

Duas por matriz plana (MinMax, Z-score) e duas por formato long, relativas ao grupo amostra/canal (pelo fundamental, pela energia espectral).

In [5]:
from harmonic_signatures.normalize.methods import (
    MinMaxNormalizer, ZScoreNormalizer, FundamentalNormalizer, SpectralEnergyNormalizer,
)

feature_cols = [c for c in harm_matrix.columns if c not in ('sample_id', 'label', 'timestamp')]
X = harm_matrix[feature_cols].to_numpy(dtype='float64')

X_minmax = MinMaxNormalizer().fit_transform(X)
X_zscore = ZScoreNormalizer().fit_transform(X)
print('minmax range:', X_minmax.min(), X_minmax.max())
print('zscore mean/std:', np.nanmean(X_zscore), np.nanstd(X_zscore))

minmax range: 0.0 1.0
zscore mean/std: -8.963529076504955e-18 0.990950040103926


In [6]:
harm_long_fundamental = FundamentalNormalizer().transform(harm_long)
harm_long_energy = SpectralEnergyNormalizer().transform(harm_long)
harm_long_fundamental.to_parquet(PROCESSED_DIR / 'harmonicas_long_norm_fundamental.parquet', index=False)
harm_long_energy.to_parquet(PROCESSED_DIR / 'harmonicas_long_norm_energy.parquet', index=False)
print('ok')

ok
